# SQL Evaluation Accuracy

This notebook loads all JSON files from the `/evaluation` folder into one DataFrame, then calculates accuracy for thinking and non-thinking responses.

It supports two cases:

1. Your JSON files already contain evaluation columns like `thinking_correct` and `non_thinking_correct`.
2. Your JSON files contain nested evaluation objects like `thinking_eval.correct` and `non_thinking_eval.correct`.


In [17]:
from pathlib import Path
import json
import pandas as pd
import numpy as np


## 1. Set paths

If your notebook is running in WSL/Linux and the project is on Windows, use the `/mnt/c/...` path.

Adjust `PROJECT_ROOT` if needed.

In [18]:
# WSL/Linux path to your project root
PROJECT_ROOT = Path(
    "/mnt/c/Users/vanes/repos/00_BFH/08_semester/00_BachelorThesis/langgraphandopenwebui/spider"
)

# If your notebook is running directly on Windows, use this instead:
# PROJECT_ROOT = Path(r"C:\Users\vanes\repos\00_BFH\08_semester\00_BachelorThesis\langgraphandopenwebui\spider")

SPLIT = "validation"
MODEL = "gemma-4"
EVALUATION_DIR = PROJECT_ROOT / "notebooks/evaluation"

print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())
print("Evaluation dir:", EVALUATION_DIR)
print("Evaluation dir exists:", EVALUATION_DIR.exists())


Project root: /mnt/c/Users/vanes/repos/00_BFH/08_semester/00_BachelorThesis/langgraphandopenwebui/spider
Project root exists: True
Evaluation dir: /mnt/c/Users/vanes/repos/00_BFH/08_semester/00_BachelorThesis/langgraphandopenwebui/spider/notebooks/evaluation
Evaluation dir exists: True


## 2. Load all JSON files into one DataFrame

In [19]:
def load_json_file(path: Path):
    """Load a JSON file that may contain either a list of records or one JSON object."""
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        # Common wrapper keys
        for key in ["data", "results", "items", "records"]:
            if key in data and isinstance(data[key], list):
                return data[key]

        # Otherwise treat the object as a single record
        return [data]

    raise ValueError(f"Unsupported JSON structure in {path}")


json_files = sorted(EVALUATION_DIR.glob("*.json"))

print(f"Found {len(json_files)} JSON files")
for file in json_files:
    print("-", file.name)

records = []

for file in json_files:
    file_records = load_json_file(file)
    for record in file_records:
        record["source_file"] = file.name
    records.extend(file_records)

df = pd.DataFrame(records)

print("Rows:", len(df))
print("Columns:", len(df.columns))
df.head()


Found 7 JSON files
- spider_test_with_ask_responses_offset_0_max_rows_100.json
- spider_test_with_ask_responses_offset_100_max_rows_200.json
- spider_test_with_ask_responses_offset_300_max_rows_100.json
- spider_test_with_ask_responses_offset_400_max_rows_100.json
- spider_test_with_ask_responses_offset_500_max_rows_100.json
- spider_test_with_ask_responses_offset_600_max_rows_200.json
- spider_test_with_ask_responses_offset_800_max_rows_234.json
Rows: 1034
Columns: 20


,db_id,query,question,query_toks,query_toks_no_value,question_toks,ask_response_thinking,ask_response_non_thinking,thinking_eval,non_thinking_eval,thinking_correct,non_thinking_correct,thinking_error,non_thinking_error,thinking_generated_sql,non_thinking_generated_sql,thinking_generated_row_count,non_thinking_generated_row_count,golden_row_count,source_file
0,concert_singer,SELECT count(*) FROM singer,How many singers do we have?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[How, many, singers, do, we, have, ?]","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm...","{'correct': True, 'error': None, 'generated_sq...","{'correct': True, 'error': None, 'generated_sq...",True,True,None,None,SELECT COUNT(singer.singer_id) AS total_singer...,SELECT COUNT(*) FROM singer,1.0,1.0,1.0,spider_test_with_ask_responses_offset_0_max_ro...
1,concert_singer,SELECT count(*) FROM singer,What is the total number of singers?,"[SELECT, count, (, *, ), FROM, singer]","[select, count, (, *, ), from, singer]","[What, is, the, total, number, of, singers, ?]","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm...","{'correct': True, 'error': None, 'generated_sq...","{'correct': True, 'error': None, 'generated_sq...",True,True,None,None,SELECT COUNT(singer.singer_id) AS total_singer...,SELECT count(*) FROM singer,1.0,1.0,1.0,spider_test_with_ask_responses_offset_0_max_ro...
2,concert_singer,"SELECT name , country , age FROM singer ORDE...","Show name, country, age for all singers ordere...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","[Show, name, ,, country, ,, age, for, all, sin...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm...","{'correct': True, 'error': None, 'generated_sq...","{'correct': True, 'error': None, 'generated_sq...",True,True,None,None,"SELECT name, country, age FROM singer ORDER BY...","SELECT\n name,\n country,\n age\nFROM singe...",6.0,6.0,6.0,spider_test_with_ask_responses_offset_0_max_ro...
3,concert_singer,"SELECT name , country , age FROM singer ORDE...","What are the names, countries, and ages for ev...","[SELECT, name, ,, country, ,, age, FROM, singe...","[select, name, ,, country, ,, age, from, singe...","[What, are, the, names, ,, countries, ,, and, ...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm...","{'correct': True, 'error': None, 'generated_sq...","{'correct': True, 'error': None, 'generated_sq...",True,True,None,None,"SELECT name, country, age FROM singer ORDER BY...","SELECT\n name,\n country,\n age\nFROM singe...",6.0,6.0,6.0,spider_test_with_ask_responses_offset_0_max_ro...
4,concert_singer,"SELECT avg(age) , min(age) , max(age) FROM s...","What is the average, minimum, and maximum age ...","[SELECT, avg, (, age, ), ,, min, (, age, ), ,,...","[select, avg, (, age, ), ,, min, (, age, ), ,,...","[What, is, the, average, ,, minimum, ,, and, m...","{'request_payload': {'user_id': 'user_123', 'm...","{'request_payload': {'user_id': 'user_123', 'm...","{'correct': True, 'error': None, 'generated_sq...","{'correct': False, 'error': 'Generated query f...",True,False,None,"Generated query failed: near ""sql"": syntax error","SELECT AVG(age) AS average_age, MIN(age) AS mi...","sql\nSELECT\n AVG(age),\n MIN(age),\n MAX(a...",1.0,NaN,1.0,spider_test_with_ask_responses_offset_0_max_ro...


## 3. Extract correctness columns

This handles these possible formats:

- `thinking_correct`
- `non_thinking_correct`
- `thinking_eval = {'correct': True}`
- `non_thinking_eval = {'correct': True}`

If your DataFrame does not yet contain evaluation results, run your SQL evaluator first and save the evaluated JSON files into `/evaluation`.

In [20]:
def get_nested_correct(value):
    """Return value['correct'] when value is a dict; otherwise NaN."""
    if isinstance(value, dict):
        return value.get("correct", np.nan)
    return np.nan


def ensure_correctness_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "thinking_correct" not in df.columns:
        if "thinking_eval" in df.columns:
            df["thinking_correct"] = df["thinking_eval"].apply(get_nested_correct)
        else:
            df["thinking_correct"] = np.nan

    if "non_thinking_correct" not in df.columns:
        if "non_thinking_eval" in df.columns:
            df["non_thinking_correct"] = df["non_thinking_eval"].apply(get_nested_correct)
        else:
            df["non_thinking_correct"] = np.nan

    # Convert true/false strings to booleans if needed
    bool_map = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }

    for col in ["thinking_correct", "non_thinking_correct"]:
        df[col] = df[col].apply(
            lambda x: bool_map.get(str(x).strip().lower(), x) if pd.notna(x) else np.nan
        )

    return df


df = ensure_correctness_columns(df)

df[["source_file", "db_id", "thinking_correct", "non_thinking_correct"]].head()


,source_file,db_id,thinking_correct,non_thinking_correct
0,spider_test_with_ask_responses_offset_0_max_ro...,concert_singer,True,True
1,spider_test_with_ask_responses_offset_0_max_ro...,concert_singer,True,True
2,spider_test_with_ask_responses_offset_0_max_ro...,concert_singer,True,True
3,spider_test_with_ask_responses_offset_0_max_ro...,concert_singer,True,True
4,spider_test_with_ask_responses_offset_0_max_ro...,concert_singer,True,False


## 4. Calculate overall accuracy

In [21]:
def accuracy(series: pd.Series) -> float:
    """Calculate mean accuracy while ignoring missing values."""
    valid = series.dropna()
    if len(valid) == 0:
        return np.nan
    return valid.astype(bool).mean()


thinking_accuracy = accuracy(df["thinking_correct"])
non_thinking_accuracy = accuracy(df["non_thinking_correct"])

summary = pd.DataFrame(
    [
        {
            "mode": "thinking",
            "correct": int(df["thinking_correct"].fillna(False).sum()),
            "evaluated": int(df["thinking_correct"].notna().sum()),
            "accuracy": thinking_accuracy,
        },
        {
            "mode": "non_thinking",
            "correct": int(df["non_thinking_correct"].fillna(False).sum()),
            "evaluated": int(df["non_thinking_correct"].notna().sum()),
            "accuracy": non_thinking_accuracy,
        },
    ]
)

summary["accuracy_percent"] = summary["accuracy"].map(
    lambda x: f"{x:.2%}" if pd.notna(x) else "N/A"
)

summary


,mode,correct,evaluated,accuracy,accuracy_percent
0,thinking,731,1034,0.706963,70.70%
1,non_thinking,620,1034,0.599613,59.96%


## 5. Accuracy by source file

In [22]:
by_file = (
    df.groupby("source_file")
    .agg(
        rows=("source_file", "size"),
        thinking_evaluated=("thinking_correct", lambda s: int(s.notna().sum())),
        thinking_correct=("thinking_correct", lambda s: int(s.fillna(False).sum())),
        thinking_accuracy=("thinking_correct", accuracy),
        non_thinking_evaluated=("non_thinking_correct", lambda s: int(s.notna().sum())),
        non_thinking_correct=("non_thinking_correct", lambda s: int(s.fillna(False).sum())),
        non_thinking_accuracy=("non_thinking_correct", accuracy),
    )
    .reset_index()
)

by_file["thinking_accuracy_percent"] = by_file["thinking_accuracy"].map(
    lambda x: f"{x:.2%}" if pd.notna(x) else "N/A"
)
by_file["non_thinking_accuracy_percent"] = by_file["non_thinking_accuracy"].map(
    lambda x: f"{x:.2%}" if pd.notna(x) else "N/A"
)

by_file


,source_file,rows,thinking_evaluated,thinking_correct,thinking_accuracy,non_thinking_evaluated,non_thinking_correct,non_thinking_accuracy,thinking_accuracy_percent,non_thinking_accuracy_percent
0,spider_test_with_ask_responses_offset_0_max_ro...,100,100,68,0.680000,100,62,0.620000,68.00%,62.00%
1,spider_test_with_ask_responses_offset_100_max_...,200,200,127,0.635000,200,118,0.590000,63.50%,59.00%
2,spider_test_with_ask_responses_offset_300_max_...,100,100,89,0.890000,100,60,0.600000,89.00%,60.00%
3,spider_test_with_ask_responses_offset_400_max_...,100,100,64,0.640000,100,56,0.560000,64.00%,56.00%
4,spider_test_with_ask_responses_offset_500_max_...,100,100,70,0.700000,100,58,0.580000,70.00%,58.00%
5,spider_test_with_ask_responses_offset_600_max_...,200,200,135,0.675000,200,99,0.495000,67.50%,49.50%
6,spider_test_with_ask_responses_offset_800_max_...,234,234,178,0.760684,234,167,0.713675,76.07%,71.37%


## 6. Accuracy by database

In [23]:
if "db_id" in df.columns:
    by_db = (
        df.groupby("db_id")
        .agg(
            rows=("db_id", "size"),
            thinking_evaluated=("thinking_correct", lambda s: int(s.notna().sum())),
            thinking_correct=("thinking_correct", lambda s: int(s.fillna(False).sum())),
            thinking_accuracy=("thinking_correct", accuracy),
            non_thinking_evaluated=("non_thinking_correct", lambda s: int(s.notna().sum())),
            non_thinking_correct=("non_thinking_correct", lambda s: int(s.fillna(False).sum())),
            non_thinking_accuracy=("non_thinking_correct", accuracy),
        )
        .reset_index()
        .sort_values("rows", ascending=False)
    )

    by_db["thinking_accuracy_percent"] = by_db["thinking_accuracy"].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "N/A"
    )
    by_db["non_thinking_accuracy_percent"] = by_db["non_thinking_accuracy"].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else "N/A"
    )

    display(by_db)
else:
    print("No db_id column found.")


,db_id,rows,thinking_evaluated,thinking_correct,thinking_accuracy,non_thinking_evaluated,non_thinking_correct,non_thinking_accuracy,thinking_accuracy_percent,non_thinking_accuracy_percent
18,world_1,120,120,62,0.516667,120,40,0.333333,51.67%,33.33%
1,car_1,92,92,31,0.336957,92,27,0.293478,33.70%,29.35%
4,cre_Doc_Template_Mgt,84,84,75,0.892857,84,49,0.583333,89.29%,58.33%
5,dog_kennels,82,82,55,0.670732,82,50,0.609756,67.07%,60.98%
7,flight_2,80,80,71,0.887500,80,67,0.837500,88.75%,83.75%
15,student_transcripts_tracking,78,78,54,0.692308,78,41,0.525641,69.23%,52.56%
19,wta_1,62,62,33,0.532258,62,27,0.435484,53.23%,43.55%
16,tvshow,62,62,47,0.758065,62,44,0.709677,75.81%,70.97%
9,network_1,56,56,45,0.803571,56,42,0.750000,80.36%,75.00%
2,concert_singer,45,45,31,0.688889,45,28,0.622222,68.89%,62.22%


## 7. Compare thinking vs non-thinking

In [24]:
comparison = df.copy()

comparison["thinking_only_correct"] = (
    comparison["thinking_correct"].eq(True)
    & comparison["non_thinking_correct"].eq(False)
)

comparison["non_thinking_only_correct"] = (
    comparison["thinking_correct"].eq(False)
    & comparison["non_thinking_correct"].eq(True)
)

comparison["both_correct"] = (
    comparison["thinking_correct"].eq(True)
    & comparison["non_thinking_correct"].eq(True)
)

comparison["both_wrong"] = (
    comparison["thinking_correct"].eq(False)
    & comparison["non_thinking_correct"].eq(False)
)

comparison_summary = pd.DataFrame(
    [
        {"category": "both_correct", "count": int(comparison["both_correct"].sum())},
        {"category": "both_wrong", "count": int(comparison["both_wrong"].sum())},
        {"category": "thinking_only_correct", "count": int(comparison["thinking_only_correct"].sum())},
        {"category": "non_thinking_only_correct", "count": int(comparison["non_thinking_only_correct"].sum())},
    ]
)

comparison_summary


,category,count
0,both_correct,584
1,both_wrong,267
2,thinking_only_correct,147
3,non_thinking_only_correct,36


## 8. Save combined results and summaries

In [25]:
OUTPUT_DIR = PROJECT_ROOT / "evaluation_summary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_DIR / "combined_evaluation_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "overall_accuracy.csv", index=False)
by_file.to_csv(OUTPUT_DIR / "accuracy_by_file.csv", index=False)

if "db_id" in df.columns:
    by_db.to_csv(OUTPUT_DIR / "accuracy_by_database.csv", index=False)

comparison_summary.to_csv(OUTPUT_DIR / "thinking_vs_non_thinking_comparison.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)
print("- combined_evaluation_results.csv")
print("- overall_accuracy.csv")
print("- accuracy_by_file.csv")
print("- accuracy_by_database.csv, if db_id exists")
print("- thinking_vs_non_thinking_comparison.csv")


Saved outputs to: /mnt/c/Users/vanes/repos/00_BFH/08_semester/00_BachelorThesis/langgraphandopenwebui/spider/evaluation_summary
- combined_evaluation_results.csv
- overall_accuracy.csv
- accuracy_by_file.csv
- accuracy_by_database.csv, if db_id exists
- thinking_vs_non_thinking_comparison.csv
